# Flipkart 5G Mobile Analytics Pipeline
This project implements an automated web-scraping pipeline and analytical engine to map competitor pricing structures, isolate technical hardware patterns, and extract actionable consumer evaluation models for 5G mobile devices under ₹50,000 on Flipkart.

## 1. Environment Ingestion & Dependency Management

In [ ]:
!pip install --upgrade pip beautifulsoup4 lxml pandas requests matplotlib seaborn

## 2. Automated Data Scraping & Safe Index Vector Balancing
We extract the target nodes per product card wrapper (`div.lvJbLV.col-12-12`) to guarantee matrix alignment and prevent index shifting due to missing elements (unrated or unpriced items).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

Products = []
Prices = []
Descriptions = []
Reviews = []

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

for i in range(1, 10):
    url = f"https://www.flipkart.com/search?q=5g+mobile+under+50000&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&page={i}"
    print(f"Scraping page {i}...")
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "lxml")
    box = soup.find("div", class_="QSCKDh dLgFEE")
    
    if box:
        cards = box.find_all("div", class_="lvJbLV col-12-12")
        for card in cards:
            name_el = card.find("div", class_="RG5Slk")
            if name_el:
                Products.append(name_el.text.strip())
                
                price_el = card.find("div", class_="hZ3P6w DeU9vF")
                Prices.append(price_el.text.strip() if price_el else None)
                
                desc_el = card.find("div", class_="CMXw7N")
                Descriptions.append(desc_el.text.strip() if desc_el else None)
                
                review_el = card.find("div", class_="MKiFS6")
                Reviews.append(float(review_el.text.strip()) if review_el else None)
    time.sleep(1)

df = pd.DataFrame({
    "Products Name": Products,
    "Prices": Prices,
    "Descriptions": Descriptions,
    "Reviews": Reviews
})
df.to_csv("Flipkart_5G_Mobile_Under_50000.csv", index=False)
print(f"Successfully saved {len(df)} aligned records to Flipkart_5G_Mobile_Under_50000.csv")

## 3. Feature Traversal & Attribute Mapping
We parse the raw specs text blocks into structural parameters (Brand, Price, RAM, ROM, Battery, Display, and Processor) using regular expression heuristics.

In [ ]:
import re

df = pd.read_csv("Flipkart_5G_Mobile_Under_50000.csv")

def extract_pricing(price_str):
    if pd.isna(price_str):
        return None
    cleaned = re.sub(r'[^\d]', '', str(price_str))
    return int(cleaned) if cleaned else None

def clean_processor(processor_str, desc_str):
    if not processor_str:
        return "Unknown"
    proc = str(processor_str).strip()
    if proc.lower() in ["octa-core", "processor", "octa core", "4 octa-core"]:
        for brand in ['Snapdragon', 'Dimensity', 'Exynos', 'Tensor', 'Bionic', 'Unisoc', 'Mediatek']:
            if brand.lower() in desc_str.lower():
                m = re.search(rf'({brand}\s*[\w\s\d®\-+]+)', desc_str, re.IGNORECASE)
                if m:
                    proc = m.group(1).strip()
                    break
    proc = re.sub(r'^(?:\d+\s+)?(?:octa-core|Yes|No)\s*', '', proc, flags=re.IGNORECASE)
    if re.search(r'^\b\d\s+Gen\s+\d\b', proc, re.IGNORECASE) or re.search(r'^\b\d[a-z]?\s+Gen\s+\d\b', proc, re.IGNORECASE) or re.search(r'^\bs\s+Gen\s+\d\b', proc, re.IGNORECASE):
        proc = "Snapdragon " + proc
    proc = re.sub(r'snapdragon', 'Snapdragon', proc, flags=re.IGNORECASE)
    proc = re.sub(r'dimensity', 'Dimensity', proc, flags=re.IGNORECASE)
    proc = re.sub(r'exynos', 'Exynos', proc, flags=re.IGNORECASE)
    proc = re.sub(r'tensor', 'Tensor', proc, flags=re.IGNORECASE)
    proc = re.sub(r'\s*(?:1\s+Year|12\s+Months|Warranty).*$', '', proc, flags=re.IGNORECASE).strip()
    proc = re.sub(r'\s*Processor\s*$', '', proc, flags=re.IGNORECASE).strip()
    return proc if proc.strip() != "" and proc.lower() not in ["yes", "no"] else "Unknown"

def parse_specs(desc):
    if pd.isna(desc):
        return {}
    desc_str = str(desc)
    specs = {}
    
    # RAM & ROM
    ram_match = re.search(r'(\d+)\s*GB\s*RAM', desc_str, re.IGNORECASE)
    specs['RAM_GB'] = int(ram_match.group(1)) if ram_match else None
    rom_match = re.search(r'(\d+)\s*GB\s*ROM', desc_str, re.IGNORECASE)
    if not rom_match:
        rom_match = re.search(r'(\d+)\s*GB\s*Storage', desc_str, re.IGNORECASE)
    specs['ROM_GB'] = int(rom_match.group(1)) if rom_match else None
    
    # Battery
    battery_match = re.search(r'(\d+)\s*mAh', desc_str, re.IGNORECASE)
    specs['Battery_mAh'] = int(battery_match.group(1)) if battery_match else None
    
    # Display
    display_match = re.search(r'(\d+\.\d+|\d+)\s*inch', desc_str, re.IGNORECASE)
    if display_match:
        specs['Display_Inches'] = float(display_match.group(1))
    else:
        display_match_cm = re.search(r'(\d+\.\d+|\d+)\s*cm', desc_str, re.IGNORECASE)
        specs['Display_Inches'] = round(float(display_match_cm.group(1)) / 2.54, 2) if display_match_cm else None
        
    # Processor
    m = re.search(r'(?:battery|Battery)(.*?)(?=Processor)', desc_str, re.IGNORECASE)
    raw_proc = m.group(1).strip() if m else None
    if not raw_proc:
        for brand in ['Snapdragon', 'Dimensity', 'Exynos', 'Tensor', 'Bionic', 'Unisoc', 'Mediatek']:
            m2 = re.search(rf'({brand}\s*[\w\s\d®\-+]+)', desc_str, re.IGNORECASE)
            if m2:
                raw_proc = m2.group(1).strip()
                break
    specs['Processor'] = clean_processor(raw_proc, desc_str)
    return specs

df['Clean_Price_INR'] = df['Prices'].apply(extract_pricing)
specs_df = df['Descriptions'].apply(parse_specs).apply(pd.Series)
df_clean = pd.concat([df, specs_df], axis=1)
df_clean['Brand'] = df_clean['Products Name'].apply(lambda x: str(x).split()[0] if pd.notna(x) else None)
df_clean['Brand'] = df_clean['Brand'].replace({'Motorola': 'Motorola/Moto', 'MOTOROLA': 'Motorola/Moto'})

df_clean.to_csv("Flipkart_5G_Mobile_Details.csv", index=False)
print("Features extracted and saved to Flipkart_5G_Mobile_Details.csv")
df_clean[['Products Name', 'Brand', 'Clean_Price_INR', 'RAM_GB', 'ROM_GB', 'Battery_mAh', 'Processor']].head()

## 4. Competitor Pricing & Market Analytics
We analyze competitor positioning by mapping pricing statistics, hardware spec distributions, and consumer rating matrices.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Competitor Price Distributions
plt.figure(figsize=(12, 6))
brand_order = df_clean['Brand'].value_counts().index[:8]
sns.boxplot(data=df_clean[df_clean['Brand'].isin(brand_order)], x='Brand', y='Clean_Price_INR', palette='Set2')
plt.title('Competitor Price Distribution for Top 5G Brands under ₹50,000')
plt.ylabel('Price (INR)')
plt.xlabel('Brand')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('competitor_price_distribution.png')
plt.show()

# 2. Rating by Brand
plt.figure(figsize=(10, 5))
brand_ratings = df_clean.groupby('Brand')['Reviews'].mean().sort_values(ascending=False).dropna()
sns.barplot(x=brand_ratings.index, y=brand_ratings.values, palette='Blues_r')
plt.title('Average Customer Rating by Brand')
plt.ylabel('Average Rating (out of 5.0)')
plt.xlabel('Brand')
plt.xticks(rotation=45)
plt.ylim(4.0, 4.7)
plt.tight_layout()
plt.savefig('brand_ratings.png')
plt.show()

# 3. RAM vs Price Heatmap
pivot_df = df_clean.pivot_table(index='RAM_GB', columns='ROM_GB', values='Clean_Price_INR', aggfunc='mean')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_df, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Average Price (INR)'})
plt.title('Average Price by Memory Configurations (RAM vs ROM)')
plt.xlabel('ROM (GB)')
plt.ylabel('RAM (GB)')
plt.tight_layout()
plt.savefig('ram_rom_price_matrix.png')
plt.show()